# PEFT LoRA 실습 : 한국어 QA Fine-tuning

**데이터** : KorQuAD 1.0 (한국어 SQuAD)
**모델** : Qwen/Qwen2.5-0.5B

학습 후 모델이 **정답만 짧게** answer 하도록 만드는 것이 목표입니다.

> ⚙️ 런타임 → 런타임 유형 변경 → **T4 GPU** 선택


## 0. 환경 설정

torchao는 PyTorch 모델 최적화·양자화 패키지이며, 이번 LoRA 실습에는 필요 없고 버전 충돌 가능성이 있어서 제거

In [ ]:
# torchao 버전 충돌 정리 (이 실습에는 불필요한 패키지)
def torchao_version():
# PEFT 정상 로드 확인

## 1. 설정값

이 셀의 값만 바꾸면 실험 조건을 조절할 수 있습니다.

## 2. 모델 · 토크나이저 로드

> **중요** : 학습할 모델은 **fp32 로 로드**합니다.
> fp16 으로 로드한 모델을 fp16 으로 학습시키면 gradient 가 제대로 반영되지 않아
> loss 가 떨어지지 않는 일이 흔합니다. 혼합 정밀도(fp16)는 Trainer 옵션으로만 켭니다.

In [ ]:
# 사전학습 모델에 맞는 토크나이저 로드
# pad_token이 정의되어 있지 않은 모델의 경우 eos_token을 padding 토큰으로 사용
# 다음 토큰 예측 방식의 GPT 계열 모델에 사용하는 클래스
# 모델 가중치를 float32 정밀도로 로드
# 모델 설정에도 tokenizer와 동일한 padding token ID 지정
# 학습 중에는 KV cache를 사용하지 않도록 비활성화

## 3. LoRA adapter 주입

LoRA는 모델 전체를 다시 학습하지 않고, 모델 안의 중요한 일부 층에만 작은 학습용 부품(adapter)을 붙여서 그 부분만 학습합니다.

여기서 target_modules는 “LoRA adapter를 어디에 붙일 것인가”를 지정하는 목록입니다.  

원본 모델은 그대로 두고 Attention과 MLP의 주요 층에 작은 LoRA 학습 모듈만 추가합니다

In [ ]:
# LoRA 학습 설정 정의
    # LoRA를 적용할 모델 내부의 Linear Layer 지정
    # q_proj, k_proj, v_proj, o_proj  → Self-Attention의 Query, Key, Value, Output projection
    # gate_proj, up_proj, down_proj  → Transformer의 FFN/MLP 영역에 있는 projection layer
# 기존 사전학습 모델에 LoRA Adapter를 삽입
# 전체 파라미터 중 실제로 학습되는 LoRA 파라미터 수와 비율 출력

## 4. 데이터 준비

### 4-1. KorQuAD 로드

### 4-2. 정답이 잘리지 않도록 문맥을 자른다  ⭐

 `answer_start` 를 이용해 **정답을 가운데 두고** 문맥을 잘라냅니다.

In [ ]:
def cut_context(context, answer_start, answer, width=CTX_CHARS):
def build(example):
# 정답이 잘린 문맥 안에 실제로 들어 있는 것만 사용

### 4-3. 토크나이징과 labels masking  ⭐

프롬프트(문맥·질문) 부분은 `-100` 으로 가려서 **정답 부분만 학습**시킵니다.
그리고 정답 끝의 **EOS 토큰은 반드시 학습 대상에 포함**시킵니다 —
이것이 빠지면 모델이 답을 끝내지 못하고 계속 지어냅니다.

In [ ]:
# 하나의 샘플(prompt, answer)을 Causal LM 학습용 토큰 형식으로 변환하는 함수
def tokenize_fn(example):
    # 프롬프트를 토큰 ID로 변환
    # add_special_tokens=False:
    # [BOS], [CLS] 등의 특수 토큰을 tokenizer가 자동으로 추가하지 않도록 설정
    # 정답(answer)을 토큰 ID로 변환
    # 앞에 공백(" ")을 추가해 prompt와 answer가 자연스럽게 이어지도록 함
    # 정답 마지막에 EOS(문장 종료) 토큰 추가
    # 모델에게 "여기서 답변 생성을 끝내라"는 패턴까지 학습시킴
    # 모델의 실제 입력:
    # prompt 토큰 + answer 토큰 + EOS 토큰
    # 학습에 사용할 정답(labels) 생성
    #
    # prompt 부분은 -100으로 지정하여 loss 계산에서 제외
    # answer 부분만 실제 토큰 ID를 정답으로 사용하여 학습
    #
    # 즉 모델은 "질문 자체를 예측"하는 것이 아니라
    # 주어진 prompt에 대해 answer를 생성하도록 학습
    # Hugging Face 모델이 사용할 입력 형태로 반환
# 전체 Dataset에 tokenize_fn을 적용하여 토큰화
# 원래의 prompt, answer 등의 컬럼은 제거하고
# input_ids, attention_mask, labels 중심의 학습 데이터로 변환
# 정답 뒷부분을 truncation하면 중요한 answer가 잘릴 수 있으므로
# 전체 길이가 MAX_LENGTH 이하인 샘플만 사용
# 최대 길이 조건을 통과한 샘플 수와 비율 출력
# 토큰 길이 분포를 간단히 확인하기 위해
# 최대 2,000개 샘플의 input_ids 길이를 계산
# 앞의 TRAIN_SIZE개 샘플을 학습 데이터로 사용
# 그 다음 EVAL_SIZE개 샘플을 평가 데이터로 사용
# 생성 결과를 사람이 직접 비교·평가하기 위해 토큰화되기 전 원본 평가 데이터를 별도로 보관

### 4-4. 정말 정답만 학습되는지 눈으로 확인

`-100` 이 아닌 토큰, 즉 **실제로 학습되는 부분**만 뽑아 봅니다.

In [ ]:
# 학습 데이터의 첫 번째 샘플 가져오기
# labels가 -100이 아닌 위치만 선택
# 전체 입력 중 마지막 80개 토큰을 사람이 읽을 수 있는 문자열로 복원해 출력
# prompt + answer가 실제로 어떻게 이어져 있는지 확인하는 용도
# 실제 학습 대상이 되는 토큰 개수 출력
# prompt 부분은 -100으로 마스킹되어 제외되고 answer + EOS 부분만 남음
# 학습 대상 토큰을 문자열로 복원해 출력
# repr()을 사용하면 공백, 개행 문자 등을 더 명확하게 확인할 수 있음
# 학습되는 마지막 토큰이 EOS 토큰인지 확인

### 4-5. Data Collator  ⭐

기본 LM collator는 우리가 만든 labels를 다시 만들 수 있어 prompt masking이 깨질 수 있습니다. 따라서 labels는 그대로 유지하고 padding만 수행하는 custom collator를 사용합니다.

In [ ]:
# QA 학습용 custom data collator 정의
# 여러 샘플의 길이를 맞춰 하나의 batch tensor로 묶는 역할
class QACollator:
    # DataLoader / Trainer가 batch를 만들 때 자동으로 호출되는 메서드
    def __call__(self, features):
        # 현재 batch 안에서 가장 긴 문장의 길이를 구함
        # padding된 input_ids, attention_mask, labels를 저장할 리스트
        # batch 안의 각 샘플을 하나씩 처리
            # input_ids는 pad_token_id로 채워 batch 내 모든 문장의 길이를 동일하게 맞춤
            # 실제 토큰 위치는 1, padding 위치는 0으로 attention_mask 구성
            # 기존 labels는 그대로 유지하고 새로 추가된 padding 위치만 -100으로 설정
# tokenizer에 설정된 pad token ID를 이용해 custom collator 객체 생성

## 5. 평가 함수 (정성 + 정량)

KorQuAD 공식 지표인 **EM(완전 일치)** 과 **F1(글자 단위 겹침)** 을 사용합니다.

In [ ]:
def batch_generate(m, prompts, max_new_tokens=32, bs=16):
# 구두점·공백 제거 후 비교용 문자열 생성
def normalize(s):
 # 예측과 정답이 완전히 같으면 1
def exact_match(p, g):
def f1_score(p, g):
def evaluate_qa(m, dataset, n=None, tag=""):

## 6. 학습 전 성능 (baseline)

사전학습 상태의 모델은 「질문에 답하는 형식」을 모르기 때문에
문맥을 그대로 이어 쓰거나 같은 말을 반복합니다.

In [ ]:
# LoRA 모델의 학습 전 성능을 평가
# 평가 데이터 중 100개 샘플을 사용하여 EM, F1, 생성 결과를 계산

## 7. 학습

Warmup step - 학습 초반에 learning rate를 바로 크게 쓰지 않고, 몇 step 동안 조금씩 올리는 구간. 학습 초반에는 모델의 gradient가 불안정할 수 있어서 처음부터 큰 learning rate를 쓰면 가중치가 갑자기 크게 변하면서 학습이 흔들릴 수 있기 때문에 사용

Colab T4 GPU 에서 약 25분 소요


In [ ]:
# 전체 학습 step 수를 계산하고 warmup step을 직접 지정
# TrainingArguments에 전달할 학습 설정을 dict 형태로 정의
# ── transformers 버전 차이를 자동으로 처리 ─────────────────────
# 구버전 transformers에서는 eval_strategy 대신 evaluation_strategy 사용
# 현재 transformers 버전에서 지원하지 않는 옵션 확인
# ────────────────────────────────────────────────────────────────
# 최종 학습 옵션 객체 생성
# Hugging Face Trainer 구성
# Fine-tuning 전에 현재 모델의 validation loss 측정
# Eval Loss를 Perplexity(PPL)로 변환
# PPL이 낮을수록 정답 토큰을 더 높은 확률로 예측하고 있다는 의미

In [ ]:
# Hugging Face Trainer를 이용해 실제 fine-tuning 학습 시작

## 8. 학습 전 / 후 비교

In [ ]:
# LoRA fine-tuning이 끝난 모델을 평가 데이터셋으로 다시 평가
# 학습 후 Eval Loss를 Perplexity(PPL)로 변환
# 학습 중 비활성화했던 KV cache를 다시 활성화
# EM, F1, 실제 생성 답변을 계산

In [ ]:
# Trainer의 학습 로그 중 training loss가 기록된 항목만 추출

## 9. 저장 · 로드 · 병합

In [ ]:
# 학습된 LoRA adapter 가중치와 설정 파일 저장
# 학습에 사용한 tokenizer 설정과 vocabulary 관련 파일 저장
# 저장된 폴더 안의 모든 파일 크기를 합산하여 전체 저장 용량 계산

### Base Model + LoRA Adapter → 하나의 일반 모델로 병합

In [ ]:
# 원본(base) Causal LM 모델을 다시 로드
# 추론 시 메모리 사용량을 줄이기 위해 float16으로 로드
# 저장해 둔 LoRA adapter를 base model에 다시 연결
# LoRA adapter 가중치를 base model 가중치에 실제로 합침 → 추론 구조가 단순해지고 속도도 원래 모델 수준에 가까워짐
# 평가 데이터의 첫 번째 prompt로 답변 생성

### 더 좋은 결과를 원한다면
- `TRAIN_SIZE` 를 20000 이상으로 (전체 60K 사용 시 EM 이 더 오릅니다)
- `MODEL_NAME` 을 `Qwen/Qwen2.5-1.5B` 로 (T4 에서도 LoRA 로 학습 가능)
- `EPOCHS` 를 5 로, `LORA_R` 을 32 로

### 참고
- [KorQuAD 공식](https://korquad.github.io/) · [PEFT 문서](https://huggingface.co/docs/peft/en/index)
